# FunSearch DEC Experiment (Cloud API, Colab)

This notebook runs FunSearch + Two-Stage DEC on Colab using a cloud API, and outputs metrics for comparison.

## Key Features
- **Same dedup implementation** as the local LLM version (Stage 1 + Stage 2).
- Optimized parameters: `stage1=15`, `stage2=256`, `max_non_code_retries=2`.
- Supports **multi-dataset mode**: each heuristic is evaluated on all datasets, providing a more realistic assessment of generalization.
- Includes metrics aligned with the proposal (Time Saved / API Efficiency / False Positive Rate / Performance Quality).


## 0) Prerequisites
1. Enable Python 3 runtime in Colab.
2. Have your repository URL and cloud API key ready.
3. If using a proxy, set `HTTPS_PROXY/HTTP_PROXY`.


In [ ]:
# 1) Clone repository
REPO_URL = "https://github.com/ultrabababa/FunSearch-DEC.git"
!git clone $REPO_URL
%cd FunSearch-DEC

In [ ]:
# 2) Install dependencies
!python -m pip install -U pip
!python -m pip install -r requirements.txt
!python -m pip install pandas matplotlib

In [ ]:
# 3) Cloud API configuration (required)
import os

os.environ['FUNSEARCH_CLOUD_API_KEY'] = '<YOUR_API_KEY>'
os.environ['FUNSEARCH_CLOUD_BASE_URL'] = 'https://api.bltcy.ai'
os.environ['FUNSEARCH_CLOUD_MODEL'] = 'gpt-5-nano'

# Disable thinking mode, output code only
os.environ['FUNSEARCH_DISABLE_THINKING'] = 'on'
os.environ['FUNSEARCH_THINKING_PARAM_MODE'] = 'both'

# Enable OpenAI SDK path (required for cloud API)
os.environ['FUNSEARCH_USE_OPENAI_SDK'] = '1'
# Minimize reasoning to speed up inference
os.environ['FUNSEARCH_REASONING_EFFORT'] = 'minimal'
os.environ['FUNSEARCH_MAX_NON_CODE_RETRIES'] = '8'
os.environ['FUNSEARCH_VERBOSE_SAMPLES'] = '1'

In [ ]:
# 4) Connectivity check (make sure this passes first)
!python tools/test_cloud_api_config.py --timeout 30

## 5) Quick Smoke Test (single dataset, repeats=1)
- Use this to verify the full pipeline works before running the full experiment.
- Uses OR_u1000 dataset (1000 items, higher eval cost, better for demonstrating DEC).


In [ ]:
# Run this cell to execute quick smoke test (repeats=1)
SMOKE_DATASET = 'OR_u1000'
SMOKE_MAX_SAMPLES = 5
SMOKE_REPEATS = 1

!python tools/run_experiment_matrix.py \
  --dataset "$SMOKE_DATASET" \
  --max-samples $SMOKE_MAX_SAMPLES \
  --repeats $SMOKE_REPEATS \
  --stage1-case-count 15 \
  --stage2-random-cases 256

!python tools/summarize_experiment_matrix.py --dataset "$SMOKE_DATASET" --repeats $SMOKE_REPEATS

## 6) Full Experiment (Multi-Dataset Mode)
- This step will:
  - Run baseline/dedup matrix
  - Evaluate each heuristic on **all datasets** (better generalization assessment)
  - Use optimized parameters: stage1=15, stage2=256
- Estimated time: ~2-4 hours (depending on API response speed)


In [ ]:
# Multi-dataset mode: all OR-Library datasets
MULTI_DATASET_KEYS = "OR_u120,OR_u250,OR_u500,OR_u1000,OR_t60,OR_t120,OR_t249,OR_t501"

!python tools/run_experiment_matrix.py \
  --dataset-keys "$MULTI_DATASET_KEYS" \
  --max-samples 20 \
  --repeats 3 \
  --stage1-case-count 15 \
  --stage2-random-cases 256

In [ ]:
# 7) Generate summary results
!python tools/summarize_experiment_matrix.py \
  --dataset-keys "$MULTI_DATASET_KEYS" \
  --repeats 3

In [ ]:
# 8) Read aggregate results
import json
from pathlib import Path
import glob

# Find the latest summary file
summary_files = sorted(glob.glob('logs/experiments/summary_*.json'))
if summary_files:
    latest = summary_files[-1]
    data = json.loads(Path(latest).read_text(encoding='utf-8'))
    print(f'\n=== {Path(latest).name} ===')
    print(json.dumps(data.get('aggregate', {}), indent=2, ensure_ascii=False))

## 9) Metrics Explanation (Aligned with Proposal)

| Proposal Metric | Field | Description |
|----------------|-------|-------------|
| **Time Saved** | `time_saved_ratio` / `pipeline_time_saved_ratio` | Evaluation time savings ratio (>0 is better) |
| **API Sample Efficiency** | `dedup_dedup_hits` / total samples | Ratio of samples intercepted by dedup (higher = more efficient) |
| **False Positive Rate** | 1 - `dedup_stage2_collision_reject_rate` | Stage 2 rejection rate (>95% is good, low false positive rate) |
| **Performance Quality** | `best_score_diff_dedup_minus_baseline` | Score difference (>0 means dedup found better heuristics) |

### Key Result Interpretation
- If `time_saved_ratio > 0`: DEC successfully saved evaluation time
- If `best_score_diff > 0`: DEC version found better heuristics
- If `stage2_collision_reject_rate > 0.95`: Stage 2 filtering is effective, low false positive rate

## 10) Local LLM Results for Comparison

The following results were obtained using a local LLM (qwen3-coder-30b-a3b-instruct) with the same experimental setup.

| Metric | Local LLM Result |
|--------|------------------|
| **Time Saved** | 58.9% (median) |
| **Pipeline Saved** | 58.8% (median) |
| **Score Improvement** | +35 to +93 (all positive) |
| **Dedup Hit Rate** | 75-80% |
| **False Positive Rate** | <5% |

**Key observations from local LLM experiments:**
1. DEC consistently found better heuristics than baseline in ALL 6 repeats
2. Average time savings of 59% across all experiments
3. Stable dedup hit rate of 75-80%
4. Stage 2 filtering effectively reduces false positives to <5%

Compare these results with your cloud API results to assess the impact of model quality on DEC effectiveness.


In [ ]:
# 11) Table and plots (quick visualization)
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import glob

# Find the latest summary CSV
csv_files = sorted(glob.glob('logs/experiments/summary_*.csv'))
if csv_files:
    latest_csv = csv_files[-1]
    df = pd.read_csv(latest_csv)
    
    print(f'=== {Path(latest_csv).name} ===')
    display(df[['repeat', 'dataset', 'baseline_best_score', 'dedup_best_score', 
                'best_score_diff_dedup_minus_baseline', 'time_saved_ratio', 
                'pipeline_time_saved_ratio', 'dedup_dedup_hits']].to_string())
    
    # Plot
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    # Time Saved
    axes[0,0].bar(df['repeat'], df['time_saved_ratio'])
    axes[0,0].axhline(0, color='gray', linestyle='--')
    axes[0,0].set_title('Time Saved Ratio')
    axes[0,0].set_xlabel('Repeat')
    axes[0,0].set_ylabel('Ratio')
    
    # Pipeline Time Saved
    axes[0,1].bar(df['repeat'], df['pipeline_time_saved_ratio'], color='tab:orange')
    axes[0,1].axhline(0, color='gray', linestyle='--')
    axes[0,1].set_title('Pipeline Time Saved Ratio')
    axes[0,1].set_xlabel('Repeat')
    axes[0,1].set_ylabel('Ratio')
    
    # Score Comparison
    x = range(len(df))
    width = 0.35
    axes[1,0].bar([i - width/2 for i in x], df['baseline_best_score'], width, label='Baseline')
    axes[1,0].bar([i + width/2 for i in x], df['dedup_best_score'], width, label='Dedup')
    axes[1,0].set_title('Best Score Comparison')
    axes[1,0].set_xlabel('Repeat')
    axes[1,0].set_ylabel('Score')
    axes[1,0].legend()
    
    # Dedup Hits
    axes[1,1].bar(df['repeat'], df['dedup_dedup_hits'], color='tab:green')
    axes[1,1].set_title('Dedup Hits')
    axes[1,1].set_xlabel('Repeat')
    axes[1,1].set_ylabel('Count')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# 12) Cloud API Results Summary (for comparison with local LLM)
import pandas as pd
import numpy as np
from pathlib import Path
import glob

# Find the latest summary CSV
csv_files = sorted(glob.glob('logs/experiments/summary_*.csv'))
if csv_files:
    latest_csv = csv_files[-1]
    df = pd.read_csv(latest_csv)
    
    print('=' * 60)
    print('Cloud API Results Summary')
    print('=' * 60)
    
    # Calculate median metrics
    median_time_saved = df['time_saved_ratio'].median()
    median_pipeline_saved = df['pipeline_time_saved_ratio'].median()
    
    # Score improvement stats
    score_diffs = df['best_score_diff_dedup_minus_baseline'].dropna()
    min_score_diff = score_diffs.min() if len(score_diffs) > 0 else 0
    max_score_diff = score_diffs.max() if len(score_diffs) > 0 else 0
    all_positive = (score_diffs > 0).all() if len(score_diffs) > 0 else False
    
    # Dedup hit rate
    total_dedup_hits = df['dedup_dedup_hits'].sum()
    # Note: total samples calculation depends on the experiment setup
    
    # Stage 2 reject rate
    stage2_reject_rates = df['dedup_stage2_collision_reject_rate'].dropna()
    avg_stage2_reject = stage2_reject_rates.mean() if len(stage2_reject_rates) > 0 else 0
    
    # Print summary table
    print(f'\nMetric                              Cloud API Result')
    print(f'-' * 50)
    print(f'Time Saved                          {median_time_saved:.1%} (median)')
    print(f'Pipeline Saved                      {median_pipeline_saved:.1%} (median)')
    print(f'Score Improvement                   {min_score_diff:+.1f} to {max_score_diff:+.1f} ({"all positive" if all_positive else "mixed"})')
    print(f'Stage 2 Reject Rate                 {avg_stage2_reject:.1%}')
    print(f'Total Dedup Hits                    {total_dedup_hits}')
    
    # Print comparison with local LLM
    print(f'\n' + '=' * 60)
    print('Comparison with Local LLM (qwen3-coder-30b-a3b-instruct)')
    print('=' * 60)
    print(f'\nMetric               Local LLM      Cloud API')
    print(f'-' * 50)
    print(f'Time Saved           58.9%          {median_time_saved:.1%}')
    print(f'Pipeline Saved       58.8%          {median_pipeline_saved:.1%}')
    print(f'Score Improvement    +35 to +93     {min_score_diff:+.1f} to {max_score_diff:+.1f}')
    print(f'Dedup Hit Rate       75-80%         See dedup_hits column')
    
    # Key observations
    print(f'\nKey observations from cloud API experiments:')
    if median_time_saved > 0:
        print(f'1. DEC saved {median_time_saved:.1%} of evaluation time on average')
    if all_positive:
        print(f'2. DEC consistently found better heuristics than baseline in ALL repeats')
    elif len(score_diffs) > 0 and (score_diffs > 0).mean() > 0.5:
        print(f'2. DEC found better heuristics in {(score_diffs > 0).mean():.0%} of repeats')
    if avg_stage2_reject > 0.95:
        print(f'3. Stage 2 filtering is effective (reject rate: {avg_stage2_reject:.1%})')
    elif avg_stage2_reject > 0:
        print(f'3. Stage 2 reject rate: {avg_stage2_reject:.1%} (target: >95%)')
    
    print(f'\n' + '=' * 60)
    print('Detailed Results per Repeat')
    print('=' * 60)
    display(df[['repeat', 'baseline_best_score', 'dedup_best_score', 
                'best_score_diff_dedup_minus_baseline', 'time_saved_ratio', 
                'pipeline_time_saved_ratio', 'dedup_dedup_hits']].to_string())

In [ ]:
# 13) Download key result files
from google.colab import files
import glob

# Download all summary files
for f in sorted(glob.glob('logs/experiments/summary_*')):
    print(f'Downloading: {f}')
    files.download(f)